# VAZHI DAPT Data Prep v2.1 — IndicAlign + Own Sources

**Key changes from v2.0:**
1. **IndicAlign Wiki_Chat** as primary corpus (~25M tokens, 62%) — 97.6% Tamil avg, diverse topics
2. **Chat replay boosted to 12%** (~5M tokens) — from OpenAssistant_T, Indic_ShareLlama, Dolly_T + local SFT
3. **WikiHow** for procedural Tamil (~3M tokens, 8%)
4. **~40M token target** (was 4.8M in v2.0 — insufficient for language acquisition)
5. **Sadhguru + classical** retained as domain-relevant content (12% + 4%)

**Why v2.0 failed:** 4.8M tokens was not enough. DAPT v1.1 (55M tokens) showed +55% Tamil improvement.
The model needs 30-50M tokens to acquire Tamil language patterns.

**Data mix (GPT5.2 reviewed):**

| Source | Target Tokens | % | Role |
|--------|--------------|---|------|
| Wiki_Chat (filtered ≥90%) | 25M | 62% | Primary Tamil prose |
| Sadhguru articles | ~5M | 12% | Domain-relevant Tamil |
| Chat replay mix | 5M | 12% | Instruction preservation |
| WikiHow | 3M | 8% | Procedural Tamil |
| Classical literature | ~1.5M | 4% | Literary Tamil, cultural |
| **Total** | **~40M** | 100% | |

```
Step 1 (THIS NOTEBOOK): Data Prep — CPU only (Colab Pro)
  -> Output: CryptoYogi/vazhi-dapt-tamil-v2_1

Step 2: DAPT Training — GPU (Colab Pro)
  -> Input:  This dataset + vanilla Qwen3-0.6B-Instruct
  -> Output: CryptoYogi/vazhi-dapt-v2_1
```

**Runtime:** ~30-60 min on CPU (streaming from HuggingFace)

In [1]:
# Cell 1 — Dependencies
!pip install -q -U \
  "transformers>=4.45.0,<5.0.0" \
  "datasets>=2.21.0" \
  "huggingface_hub>=0.24.7"

print("\u2705 Dependencies installed (CPU-only)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.4 MB/s eta 0:00:00
✅ Dependencies installed (CPU-only)


In [2]:
# Cell 2 — Configuration
import os
import re
import json
import random
import hashlib
import unicodedata
import numpy as np
from collections import Counter

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# === KEY CONFIG ===
HF_DATASET_OUT = "CryptoYogi/vazhi-dapt-tamil-v2_1"
MODEL_ID = "Qwen/Qwen3-0.6B"  # For tokenizer only
BLOCK_SIZE = 1024

# === DATA QUALITY FILTERS ===
TAMIL_THRESHOLD = 0.90  # >= 90% Tamil chars
MIN_CHARS = 100
MAX_CHARS = 50000
MAX_REPETITION_RATIO = 0.5

# === TOKEN BUDGET TARGETS ===
# Qwen3 tokenizer: ~1 token/char for Tamil (measured in v2.0)
TARGET_WIKI_CHAT_TOKENS = 25_000_000     # 62%
TARGET_CHAT_REPLAY_TOKENS = 5_000_000    # 12% — critical for instruction preservation
TARGET_WIKIHOW_TOKENS = 3_000_000        # 8%
# Sadhguru + classical: take all (no cap needed, ~5M + ~0.4M)

# === INDICALIGN CONFIG ===
INDICALIGN_REPO = "ai4bharat/indic-align"
TAMIL_COLUMN = "tam_Taml"  # Tamil content column in IndicAlign

# === LOCAL FILE SOURCES (uploaded to Colab or downloaded from HF) ===
SOURCES_REPO = "CryptoYogi/vazhi-dapt-sources-v2_0"  # Reuse v2.0 source files
SADHGURU_PATH = "articles_filtered_full.json"
DAPT_CORPUS_FILES = {
    "thirukkural": "37_thirukkural_corpus.json",
    "bharathiar": "40_bharathiar_corpus.json",
    "silapathikaram": "36_silapathikaram_corpus.json",
    "sangam": "38_sangam_corpus.json",
    "aathichoodi": "39_aathichoodi_corpus.json",
}
LOCAL_CHAT_FILES = [
    "conversational_fundamentals.json",
    "vazhi_behavior_pack.json",
]

print(f"\U0001f4cb DAPT Data Prep v2.1 Config:")
print(f"   Tokenizer:        {MODEL_ID}")
print(f"   Output:           {HF_DATASET_OUT}")
print(f"   Block size:       {BLOCK_SIZE} tokens")
print(f"   Tamil threshold:  >= {TAMIL_THRESHOLD:.0%}")
print(f"   Token targets:")
print(f"     Wiki_Chat:      {TARGET_WIKI_CHAT_TOKENS:>12,} (62%)")
print(f"     Chat replay:    {TARGET_CHAT_REPLAY_TOKENS:>12,} (12%)")
print(f"     WikiHow:        {TARGET_WIKIHOW_TOKENS:>12,} (8%)")
print(f"     Sadhguru:       all (~5M, 12%)")
print(f"     Classical:      all (~0.4M, 4%)")
print(f"   Total target:     ~40M tokens")

📋 DAPT Data Prep v2.1 Config:
   Tokenizer:        Qwen/Qwen3-0.6B
   Output:           CryptoYogi/vazhi-dapt-tamil-v2_1
   Block size:       1024 tokens
   Tamil threshold:  >= 90%
   Token targets:
     Wiki_Chat:        25,000,000 (62%)
     Chat replay:       5,000,000 (12%)
     WikiHow:           3,000,000 (8%)
     Sadhguru:       all (~5M, 12%)
     Classical:      all (~0.4M, 4%)
   Total target:     ~40M tokens


In [3]:
# Cell 3 — HuggingFace Login
from huggingface_hub import notebook_login
notebook_login()
print("\u2705 Logged in to HuggingFace")

✅ Logged in to HuggingFace


In [4]:
# Cell 4 — Download Local Source Files from HuggingFace
from huggingface_hub import hf_hub_download

FILES = [
    SADHGURU_PATH,
    *DAPT_CORPUS_FILES.values(),
    *LOCAL_CHAT_FILES,
]

print(f"\U0001f4e5 Downloading {len(FILES)} source files from {SOURCES_REPO}...")
for fname in FILES:
    if os.path.exists(fname):
        print(f"   \u2705 {fname} (already present)")
        continue
    path = hf_hub_download(
        repo_id=SOURCES_REPO, filename=fname,
        repo_type="dataset", local_dir=".",
    )
    size = os.path.getsize(fname)
    print(f"   \u2705 {fname} ({size:,} bytes)")

print(f"\n\u2705 All {len(FILES)} local source files ready")

📥 Downloading 8 source files from CryptoYogi/vazhi-dapt-sources-v2_0...


articles_filtered_full.json:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

   ✅ articles_filtered_full.json (10,890,888 bytes)


37_thirukkural_corpus.json: 0.00B [00:00, ?B/s]

   ✅ 37_thirukkural_corpus.json (966,431 bytes)


...

   ✅ 39_aathichoodi_corpus.json (81,448 bytes)


conversational_fundamentals.json: 0.00B [00:00, ?B/s]

   ✅ conversational_fundamentals.json (120,082 bytes)


vazhi_behavior_pack.json: 0.00B [00:00, ?B/s]

   ✅ vazhi_behavior_pack.json (64,764 bytes)

✅ All 8 local source files ready


In [5]:
# Cell 5 — Load Tokenizer
from transformers import AutoTokenizer

print(f"\U0001f4e5 Loading tokenizer from {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"\u2705 Tokenizer ready: {len(tokenizer)} tokens")
print(f"   eos_token: {tokenizer.eos_token!r} (ID {tokenizer.eos_token_id})")

# Verify ~1 tok/char for Tamil (lesson from v2.0)
test_tamil = "\u0ba4\u0bae\u0bbf\u0bb4\u0bcd \u0ba8\u0bbe\u0b9f\u0bc1 \u0b85\u0bb4\u0b95\u0bbe\u0ba9 \u0bae\u0bbe\u0ba8\u0bbf\u0bb2\u0bae\u0bcd"
test_tokens = tokenizer.encode(test_tamil, add_special_tokens=False)
ratio = len(test_tokens) / len(test_tamil)
print(f"   Token/char ratio: {ratio:.2f} ('{test_tamil}' -> {len(test_tokens)} tokens)")

📥 Loading tokenizer from Qwen/Qwen3-0.6B...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

✅ Tokenizer ready: 151669 tokens
   eos_token: '<|im_end|>' (ID 151645)
   Token/char ratio: 1.17 ('தமிழ் நாடு அழகான மாநிலம்' -> 28 tokens)


In [6]:
# Cell 6 — Cleaning & Quality Functions

# Zero-width and invisible characters to strip
_INVISIBLE_RE = re.compile(
    '['
    '\u200b'  # zero-width space
    '\u200c'  # zero-width non-joiner
    '\u200d'  # zero-width joiner
    '\u200e'  # left-to-right mark
    '\u200f'  # right-to-left mark
    '\u00ad'  # soft hyphen
    '\ufeff'  # byte order mark
    '\u2060'  # word joiner
    '\u2061\u2062\u2063\u2064'  # invisible operators
    '\ufff9\ufffa\ufffb'  # interlinear annotations
    ']'
)

# Control characters (keep newline, tab, carriage return)
_CONTROL_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]')


def nfkc_normalize(text):
    """NFKC normalize + strip invisible/control chars + collapse whitespace."""
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\ufffd', '')
    text = _INVISIBLE_RE.sub('', text)
    text = _CONTROL_RE.sub('', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' *\n *', '\n', text)
    return text.strip()


def clean_article_text(text):
    """Remove HTML/markdown artifacts from scraped Sadhguru article text."""
    text = re.sub(r'\[/?[Ss]adhguru[Ii]mage[^\]]*\]', '', text)
    text = re.sub(r'\[/?[Pp]ull[Qq]uote\s*\]', '', text)
    text = re.sub(r'\[[Ss]eparator[^\]]*\]', '', text)
    text = re.sub(r'\[/?[Pp]hoto[Cc]redit[^\]]*\]', '', text)
    text = re.sub(r'\[/?[A-Za-z]+[^\]]*\]', '', text)
    text = re.sub(r'\n\s*\u0b95\u0bc1\u0bb1\u0bbf\u0baa\u0bcd\u0baa\u0bc1\s*:.*$', '', text, flags=re.DOTALL)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text)
    # Remove lines that are just quoted text markers
    SINGLE_QUOTE_LINE = "^'[^']+' *$"
    text = re.sub(SINGLE_QUOTE_LINE, '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'  +', ' ', text)
    return text.strip()


def tamil_char_pct(text):
    """Compute Tamil Unicode char % (among non-whitespace, non-digit chars)."""
    if not text:
        return 0.0
    tamil = sum(1 for c in text if '\u0B80' <= c <= '\u0BFF')
    total = sum(1 for c in text if not c.isspace() and not c.isdigit())
    return tamil / total if total > 0 else 0.0


def has_excessive_repetition(text, threshold=MAX_REPETITION_RATIO):
    """Detect repeated lines (boilerplate)."""
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) < 3:
        return False
    line_counts = Counter(lines)
    most_common_count = line_counts.most_common(1)[0][1]
    return most_common_count / len(lines) > threshold


def text_hash(text):
    """MD5 hash for dedup."""
    return hashlib.md5(text.encode('utf-8')).hexdigest()


def extract_tamil_from_indicalign(row, column=TAMIL_COLUMN):
    """Extract Tamil text from an IndicAlign row.
    Returns list of text strings (one per conversation turn).
    Format: [[question, answer], [question, answer], ...]
    """
    tamil_data = row.get(column)
    if not tamil_data or not isinstance(tamil_data, list):
        return []
    texts = []
    for turn in tamil_data:
        if isinstance(turn, list) and len(turn) >= 2:
            # For raw prose (DAPT), concatenate Q+A as continuous text
            combined = turn[0].strip() + '\n\n' + turn[1].strip()
            texts.append(combined)
    return texts


def extract_chatml_from_indicalign(row, column=TAMIL_COLUMN):
    """Extract Tamil as ChatML-formatted text from an IndicAlign row.
    For chat replay during DAPT — preserves instruction-following structure.
    """
    tamil_data = row.get(column)
    if not tamil_data or not isinstance(tamil_data, list):
        return []
    texts = []
    for turn in tamil_data:
        if isinstance(turn, list) and len(turn) >= 2:
            q, a = turn[0].strip(), turn[1].strip()
            if q and a:
                chatml = (
                    f"<|im_start|>user\n{q}<|im_end|>\n"
                    f"<|im_start|>assistant\n{a}<|im_end|>"
                )
                texts.append(chatml)
    return texts


# Quick test
test_text = "  \u200b\u0ba4\u0bae\u0bbf\u0bb4\u0bcd\ufffd  \u0ba8\u0bbe\u0b9f\u0bc1  \n\n\n\n  test  "
cleaned = nfkc_normalize(test_text)
print(f"\u2705 Cleaning pipeline ready")
print(f"   Test: {test_text!r}")
print(f"   Clean: {cleaned!r}")
assert '\ufffd' not in cleaned
assert '\u200b' not in cleaned
print(f"   All assertions passed")

✅ Cleaning pipeline ready
   Test: '  \u200bதமிழ்�  நாடு  \n\n\n\n  test  '
   Clean: 'தமிழ் நாடு\n\ntest'
   All assertions passed


In [7]:
# Cell 7 — Process Wiki_Chat (Primary Source, ~62% of tokens)
# Stream from IndicAlign, filter >=90% Tamil, cap at TARGET_WIKI_CHAT_TOKENS

from datasets import load_dataset

print(f"\U0001f4e5 Streaming Wiki_Chat from {INDICALIGN_REPO}...")
print(f"   Target: {TARGET_WIKI_CHAT_TOKENS:,} tokens")

ds_wiki = load_dataset(INDICALIGN_REPO, 'Wiki_Chat', split='train', streaming=True)

wiki_chat_texts = []
wiki_chat_tokens_est = 0
wiki_chat_stats = {'docs_scanned': 0, 'texts_kept': 0, 'texts_dropped_tamil': 0,
                   'texts_dropped_short': 0, 'texts_dropped_repetition': 0,
                   'texts_dropped_dedup': 0}
seen_hashes_wiki = set()

# Track Tamil % for reporting
wiki_tamil_pcts = []

for row in ds_wiki:
    wiki_chat_stats['docs_scanned'] += 1

    # Extract Tamil conversation turns as raw prose
    texts = extract_tamil_from_indicalign(row)

    for text in texts:
        text = nfkc_normalize(text)

        if len(text) < MIN_CHARS:
            wiki_chat_stats['texts_dropped_short'] += 1
            continue

        pct = tamil_char_pct(text)
        if pct < TAMIL_THRESHOLD:
            wiki_chat_stats['texts_dropped_tamil'] += 1
            continue

        if has_excessive_repetition(text):
            wiki_chat_stats['texts_dropped_repetition'] += 1
            continue

        h = text_hash(text)
        if h in seen_hashes_wiki:
            wiki_chat_stats['texts_dropped_dedup'] += 1
            continue
        seen_hashes_wiki.add(h)

        wiki_chat_texts.append(text)
        wiki_chat_stats['texts_kept'] += 1
        wiki_tamil_pcts.append(pct)

        # Estimate tokens (~1 tok/char for Qwen3 Tamil)
        wiki_chat_tokens_est += len(text)

    # Progress reporting
    if wiki_chat_stats['docs_scanned'] % 2000 == 0:
        print(f"   ... {wiki_chat_stats['docs_scanned']:,} docs scanned, "
              f"{wiki_chat_stats['texts_kept']:,} texts kept, "
              f"~{wiki_chat_tokens_est:,} est. tokens")

    # Stop when we have enough tokens
    if wiki_chat_tokens_est >= TARGET_WIKI_CHAT_TOKENS:
        print(f"\n   \U0001f3af Reached token target at {wiki_chat_stats['docs_scanned']:,} docs")
        break

# Compute actual token count on sample for accuracy check
sample_size = min(200, len(wiki_chat_texts))
sample_token_count = sum(len(tokenizer.encode(t, add_special_tokens=False))
                         for t in wiki_chat_texts[:sample_size])
actual_ratio = sample_token_count / sum(len(t) for t in wiki_chat_texts[:sample_size])
actual_wiki_tokens = int(wiki_chat_tokens_est * actual_ratio)

print(f"\n\u2705 Wiki_Chat processed:")
print(f"   Docs scanned:     {wiki_chat_stats['docs_scanned']:,}")
print(f"   Texts kept:       {wiki_chat_stats['texts_kept']:,}")
print(f"   Dropped (tamil):  {wiki_chat_stats['texts_dropped_tamil']:,}")
print(f"   Dropped (short):  {wiki_chat_stats['texts_dropped_short']:,}")
print(f"   Dropped (repeat): {wiki_chat_stats['texts_dropped_repetition']:,}")
print(f"   Dropped (dedup):  {wiki_chat_stats['texts_dropped_dedup']:,}")
print(f"   Token/char ratio: {actual_ratio:.3f}")
print(f"   Est. tokens:      {wiki_chat_tokens_est:,} (char-based)")
print(f"   Act. tokens:      {actual_wiki_tokens:,} (tokenizer-verified)")
if wiki_tamil_pcts:
    print(f"   Tamil %: avg={np.mean(wiki_tamil_pcts):.1%}, "
          f"min={min(wiki_tamil_pcts):.1%}, max={max(wiki_tamil_pcts):.1%}")

# Spot check
if wiki_chat_texts:
    sample = random.choice(wiki_chat_texts)
    print(f"\n\U0001f4d6 Sample (Tamil {tamil_char_pct(sample):.0%}, {len(sample)} chars):")
    print(f"   {sample[:300]}...")

📥 Streaming Wiki_Chat from ai4bharat/indic-align...
   Target: 25,000,000 tokens


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

   ... 2,000 docs scanned, 5,310 texts kept, ~9,648,986 est. tokens
   ... 4,000 docs scanned, 10,745 texts kept, ~19,611,681 est. tokens

   🎯 Reached token target at 5,083 docs

✅ Wiki_Chat processed:
   Docs scanned:     5,083
   Texts kept:       13,649
   Dropped (tamil):  85
   Dropped (short):  119
   Dropped (repeat): 0
   Dropped (dedup):  92
   Token/char ratio: 1.090
   Est. tokens:      25,001,492 (char-based)
   Act. tokens:      27,247,265 (tokenizer-verified)
   Tamil %: avg=97.8%, min=90.0%, max=99.4%

📖 Sample (Tamil 93%, 1946 chars):
   "பிரம்மகுப்தர் உருவாக்கிய இடைக்கணிப்பு சூத்திரம் மற்றும் அந்த கால வானியலாளர்கள் பயன்படுத்திய பிற முறைகளிலிருந்து இது எவ்வாறு வேறுபடுகிறது என்பது பற்றிய கூடுதல் தகவல்களை வழங்க முடியுமா?

பிரம்மகுப்தரின் இடைக்கணிப்பு சூத்திரம், பிரம்மகுப்த-நியூட்டன் இடைக்கணிப்பு சூத்திரம் என்றும் அழைக்கப்படுகிறது, இது...


In [8]:
# Cell 8 — Process Sadhguru Articles (Domain Source, ~12%)

print(f"\U0001f4e5 Loading Sadhguru articles from {SADHGURU_PATH}...")
with open(SADHGURU_PATH, 'r', encoding='utf-8') as f:
    articles = json.load(f)
print(f"   Loaded {len(articles)} articles")

sadhguru_texts = []
sadhguru_stats = {'kept': 0, 'dropped_short': 0, 'dropped_tamil': 0,
                  'dropped_repetition': 0, 'dropped_dedup': 0}
seen_hashes_sg = set()

for article in articles:
    raw_text = article.get('tamil_text', '')
    if not raw_text:
        sadhguru_stats['dropped_short'] += 1
        continue

    text = clean_article_text(raw_text)
    text = nfkc_normalize(text)

    if len(text) < MIN_CHARS:
        sadhguru_stats['dropped_short'] += 1
        continue
    pct = tamil_char_pct(text)
    if pct < TAMIL_THRESHOLD:
        sadhguru_stats['dropped_tamil'] += 1
        continue
    if has_excessive_repetition(text):
        sadhguru_stats['dropped_repetition'] += 1
        continue
    h = text_hash(text)
    if h in seen_hashes_sg:
        sadhguru_stats['dropped_dedup'] += 1
        continue
    seen_hashes_sg.add(h)

    sadhguru_texts.append(text)
    sadhguru_stats['kept'] += 1

# Token estimation
sample_size = min(50, len(sadhguru_texts))
sample_tokens = sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in sadhguru_texts[:sample_size])
avg_tokens_per_doc = sample_tokens / max(sample_size, 1)
est_sadhguru_tokens = int(avg_tokens_per_doc * len(sadhguru_texts))

print(f"\n\u2705 Sadhguru articles processed:")
print(f"   Kept:             {sadhguru_stats['kept']}")
print(f"   Dropped (short):  {sadhguru_stats['dropped_short']}")
print(f"   Dropped (tamil):  {sadhguru_stats['dropped_tamil']}")
print(f"   Dropped (repeat): {sadhguru_stats['dropped_repetition']}")
print(f"   Dropped (dedup):  {sadhguru_stats['dropped_dedup']}")
print(f"   Avg tokens/doc:   {avg_tokens_per_doc:.0f}")
print(f"   Est. total tokens: {est_sadhguru_tokens:,}")

📥 Loading Sadhguru articles from articles_filtered_full.json...
   Loaded 562 articles

✅ Sadhguru articles processed:
   Kept:             561
   Dropped (short):  0
   Dropped (tamil):  1
   Dropped (repeat): 0
   Dropped (dedup):  0
   Avg tokens/doc:   6912
   Est. total tokens: 3,877,822


In [9]:
# Cell 9 — Process Chat Replay (Instruction Preservation, ~12%)
# Sources: IndicAlign (OpenAssistant_T, Indic_ShareLlama, Dolly_T) + local SFT data
# ALL formatted as ChatML to preserve instruction-following during DAPT

chat_replay_texts = []
chat_replay_tokens_est = 0
chat_replay_stats = {}
seen_hashes_chat = set()

# --- Part A: IndicAlign conversational subsets ---
CHAT_SUBSETS = [
    ('OpenAssistant_T', 2_000_000),   # ~2M tokens — explanatory conversations
    ('Indic_ShareLlama', 1_500_000),  # ~1.5M tokens — conversational/roleplay
    ('Dolly_T', 500_000),             # ~0.5M tokens — short Q&A
]

for config_name, token_target in CHAT_SUBSETS:
    print(f"\U0001f4e5 Streaming {config_name} (target: {token_target:,} tokens)...")

    ds = load_dataset(INDICALIGN_REPO, config_name, split='train', streaming=True)

    subset_kept = 0
    subset_dropped = 0
    subset_tokens = 0

    for row in ds:
        # Extract as ChatML
        chatml_texts = extract_chatml_from_indicalign(row)

        for text in chatml_texts:
            # Check Tamil quality on the content (strip ChatML tags for quality check)
            content_only = re.sub(r'<\|im_start\|>\w+\n|<\|im_end\|>', '', text)
            content_only = nfkc_normalize(content_only)

            if len(content_only) < 50:
                subset_dropped += 1
                continue

            pct = tamil_char_pct(content_only)
            if pct < TAMIL_THRESHOLD:
                subset_dropped += 1
                continue

            h = text_hash(content_only)
            if h in seen_hashes_chat:
                subset_dropped += 1
                continue
            seen_hashes_chat.add(h)

            chat_replay_texts.append(text)
            subset_kept += 1
            subset_tokens += len(text)  # ~1 tok/char estimate
            chat_replay_tokens_est += len(text)

        # Stop this subset when target reached
        if subset_tokens >= token_target:
            break

    chat_replay_stats[config_name] = {'kept': subset_kept, 'dropped': subset_dropped,
                                       'est_tokens': subset_tokens}
    print(f"   \u2705 {config_name}: {subset_kept:,} texts, ~{subset_tokens:,} tokens")

# --- Part B: Local SFT data as ChatML ---
local_chat_count = 0
local_chat_tokens = 0

for chat_file in LOCAL_CHAT_FILES:
    if not os.path.exists(chat_file):
        print(f"   \u26a0\ufe0f {chat_file} not found. Skipping.")
        continue
    with open(chat_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for item in data:
        instruction = item.get('instruction', '')
        output = item.get('output', '')
        if not instruction or not output:
            continue
        chatml = (
            f"<|im_start|>user\n{instruction}<|im_end|>\n"
            f"<|im_start|>assistant\n{output}<|im_end|>"
        )
        chat_replay_texts.append(chatml)
        local_chat_count += 1
        local_chat_tokens += len(chatml)
        chat_replay_tokens_est += len(chatml)

chat_replay_stats['local_sft'] = {'kept': local_chat_count, 'est_tokens': local_chat_tokens}
print(f"   \u2705 Local SFT: {local_chat_count} texts, ~{local_chat_tokens:,} tokens")

# Verify actual token count on sample
sample_size = min(200, len(chat_replay_texts))
sample_tok = sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in chat_replay_texts[:sample_size])
chat_actual_ratio = sample_tok / sum(len(t) for t in chat_replay_texts[:sample_size])
actual_chat_tokens = int(chat_replay_tokens_est * chat_actual_ratio)

print(f"\n\U0001f4ca Chat Replay Summary:")
print(f"   Total texts:     {len(chat_replay_texts):,}")
print(f"   Est. tokens:     {chat_replay_tokens_est:,} (char-based)")
print(f"   Act. tokens:     {actual_chat_tokens:,} (tokenizer-verified)")
print(f"   Token/char ratio: {chat_actual_ratio:.3f}")
for name, stats in chat_replay_stats.items():
    print(f"     {name}: {stats['kept']:,} kept, ~{stats['est_tokens']:,} est. tokens")

# Show sample
if chat_replay_texts:
    sample = random.choice(chat_replay_texts)
    print(f"\n\U0001f4d6 Sample ChatML:")
    print(f"   {sample[:300]}...")

📥 Streaming OpenAssistant_T (target: 2,000,000 tokens)...
   ✅ OpenAssistant_T: 1,818 texts, ~2,000,598 tokens
📥 Streaming Indic_ShareLlama (target: 1,500,000 tokens)...
   ✅ Indic_ShareLlama: 590 texts, ~1,500,141 tokens
📥 Streaming Dolly_T (target: 500,000 tokens)...
   ✅ Dolly_T: 815 texts, ~500,193 tokens
   ✅ Local SFT: 384 texts, ~79,115 tokens

📊 Chat Replay Summary:
   Total texts:     3,607
   Est. tokens:     4,080,047 (char-based)
   Act. tokens:     4,233,178 (tokenizer-verified)
   Token/char ratio: 1.038
     OpenAssistant_T: 1,818 kept, ~2,000,598 est. tokens
     Indic_ShareLlama: 590 kept, ~1,500,141 est. tokens
     Dolly_T: 815 kept, ~500,193 est. tokens
     local_sft: 384 kept, ~79,115 est. tokens

📖 Sample ChatML:
   <|im_start|>user
அமெரிக்கப் புரட்சியின் முக்கியத்துவம், அதற்கு வழிவகுத்த நிகழ்வுகள், உலகில் அது ஏற்படுத்திய தாக்கம் மற்றும் இன்று அதன் தொடர் பொருத்தம் ஆகியவற்றை விளக்குங்கள்.<|im_end|>
<|im_start|>assistant
அமெரிக்கப் புரட்சி என்பது உலக வரலாற்றில் ஒரு

In [10]:
# Cell 10 — Process WikiHow (Procedural Tamil, ~8%)

print(f"\U0001f4e5 Streaming WikiHow from {INDICALIGN_REPO}...")
print(f"   Target: {TARGET_WIKIHOW_TOKENS:,} tokens")

ds_wikihow = load_dataset(INDICALIGN_REPO, 'WikiHow', split='train', streaming=True)

wikihow_texts = []
wikihow_tokens_est = 0
wikihow_stats = {'docs_scanned': 0, 'texts_kept': 0, 'texts_dropped': 0}
seen_hashes_wh = set()

for row in ds_wikihow:
    wikihow_stats['docs_scanned'] += 1

    # Extract as raw prose (not ChatML — WikiHow is procedural content)
    texts = extract_tamil_from_indicalign(row)

    for text in texts:
        text = nfkc_normalize(text)

        if len(text) < MIN_CHARS:
            wikihow_stats['texts_dropped'] += 1
            continue

        pct = tamil_char_pct(text)
        if pct < TAMIL_THRESHOLD:
            wikihow_stats['texts_dropped'] += 1
            continue

        h = text_hash(text)
        if h in seen_hashes_wh:
            wikihow_stats['texts_dropped'] += 1
            continue
        seen_hashes_wh.add(h)

        wikihow_texts.append(text)
        wikihow_stats['texts_kept'] += 1
        wikihow_tokens_est += len(text)

    if wikihow_stats['docs_scanned'] % 500 == 0:
        print(f"   ... {wikihow_stats['docs_scanned']:,} docs, "
              f"{wikihow_stats['texts_kept']:,} kept, ~{wikihow_tokens_est:,} tokens")

    if wikihow_tokens_est >= TARGET_WIKIHOW_TOKENS:
        print(f"\n   \U0001f3af Reached token target at {wikihow_stats['docs_scanned']:,} docs")
        break

# Verify token count
sample_size = min(100, len(wikihow_texts))
if sample_size > 0:
    sample_tok = sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in wikihow_texts[:sample_size])
    wh_ratio = sample_tok / sum(len(t) for t in wikihow_texts[:sample_size])
    actual_wh_tokens = int(wikihow_tokens_est * wh_ratio)
else:
    wh_ratio = 1.0
    actual_wh_tokens = 0

print(f"\n\u2705 WikiHow processed:")
print(f"   Docs scanned:  {wikihow_stats['docs_scanned']:,}")
print(f"   Texts kept:    {wikihow_stats['texts_kept']:,}")
print(f"   Texts dropped: {wikihow_stats['texts_dropped']:,}")
print(f"   Est. tokens:   {wikihow_tokens_est:,} (char-based)")
print(f"   Act. tokens:   {actual_wh_tokens:,} (tokenizer-verified)")

📥 Streaming WikiHow from ai4bharat/indic-align...
   Target: 3,000,000 tokens
   ... 500 docs, 496 kept, ~2,486,097 tokens

   🎯 Reached token target at 610 docs

✅ WikiHow processed:
   Docs scanned:  610
   Texts kept:    606
   Texts dropped: 4
   Est. tokens:   3,010,542 (char-based)
   Act. tokens:   3,289,273 (tokenizer-verified)


In [11]:
# Cell 11 — Process Classical Literature (~4%)

classical_texts = []
classical_stats = {}

def process_thirukkural(data):
    texts = []
    for item in data:
        verse = item.get('tamil', '')
        meaning = item.get('meaning_tamil', '')
        if verse and meaning:
            texts.append(f"{verse} {meaning}")
    return texts

def process_bharathiar(data):
    texts = []
    poems = data.get('poems', [])
    for poem in poems:
        text = poem.get('full_text', '')
        if text:
            texts.append(text)
    return texts

def process_silapathikaram(data):
    return [item.get('text', '') for item in data if item.get('text')]

def process_sangam(data):
    return [item.get('text', '') for item in data if item.get('text')]

def process_aathichoodi(data):
    texts = []
    for section_key in ['aathichoodi', 'konrai_venthan']:
        section = data.get(section_key, {})
        verses = section.get('verses', [])
        for v in verses:
            verse = v.get('verse', '')
            meaning = v.get('meaning_tamil', '')
            if verse and meaning:
                texts.append(f"{verse} - {meaning}")
            elif verse:
                texts.append(verse)
    return texts

PROCESSORS = {
    'thirukkural': process_thirukkural,
    'bharathiar': process_bharathiar,
    'silapathikaram': process_silapathikaram,
    'sangam': process_sangam,
    'aathichoodi': process_aathichoodi,
}

for name, filename in DAPT_CORPUS_FILES.items():
    print(f"\U0001f4e5 Processing {name} ({filename})...")
    if not os.path.exists(filename):
        print(f"   \u26a0\ufe0f File not found: {filename}. Skipping.")
        classical_stats[name] = {'kept': 0, 'dropped': 0}
        continue
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    processor = PROCESSORS[name]
    raw_texts = processor(data)
    kept = 0
    dropped = 0
    for text in raw_texts:
        text = nfkc_normalize(text)
        if not text or len(text) < 10:
            dropped += 1
            continue
        pct = tamil_char_pct(text)
        if pct < 0.80:  # Relaxed for classical (spaces, punctuation)
            dropped += 1
            continue
        classical_texts.append(text)
        kept += 1
    classical_stats[name] = {'kept': kept, 'dropped': dropped}
    print(f"   \u2705 {name}: kept {kept}, dropped {dropped}")

# Estimate classical tokens
if classical_texts:
    sample_size = min(100, len(classical_texts))
    sample_tokens = sum(len(tokenizer.encode(t, add_special_tokens=False)) for t in classical_texts[:sample_size])
    est_classical_tokens = int(sample_tokens / sample_size * len(classical_texts))
else:
    est_classical_tokens = 0

print(f"\n\U0001f4ca Classical literature summary:")
print(f"   Total texts: {len(classical_texts)}")
print(f"   Est. tokens: {est_classical_tokens:,}")
for name, stats in classical_stats.items():
    print(f"   {name}: kept {stats['kept']}, dropped {stats['dropped']}")

📥 Processing thirukkural (37_thirukkural_corpus.json)...
   ✅ thirukkural: kept 1330, dropped 0
📥 Processing bharathiar (40_bharathiar_corpus.json)...
   ✅ bharathiar: kept 109, dropped 0
📥 Processing silapathikaram (36_silapathikaram_corpus.json)...
   ✅ silapathikaram: kept 142, dropped 0
📥 Processing sangam (38_sangam_corpus.json)...
   ✅ sangam: kept 63, dropped 0
📥 Processing aathichoodi (39_aathichoodi_corpus.json)...
   ✅ aathichoodi: kept 200, dropped 0

📊 Classical literature summary:
   Total texts: 1844
   Est. tokens: 359,635
   thirukkural: kept 1330, dropped 0
   bharathiar: kept 109, dropped 0
   silapathikaram: kept 142, dropped 0
   sangam: kept 63, dropped 0
   aathichoodi: kept 200, dropped 0


In [12]:
# Cell 12 — Quality Verification Gate

# Compute Tamil % on prose texts (Wiki_Chat + Sadhguru + WikiHow + classical)
all_prose_texts = wiki_chat_texts + sadhguru_texts + wikihow_texts + classical_texts
all_chat_texts = chat_replay_texts

# Sample prose for Tamil % check
sample_size = min(500, len(all_prose_texts))
sample_indices = random.sample(range(len(all_prose_texts)), sample_size)
sample_pcts = [tamil_char_pct(all_prose_texts[i]) for i in sample_indices]
avg_tamil = np.mean(sample_pcts)

# Token estimates
est_total_prose = actual_wiki_tokens + est_sadhguru_tokens + actual_wh_tokens + est_classical_tokens
est_total = est_total_prose + actual_chat_tokens

print(f"{'='*65}")
print(f"\U0001f4ca QUALITY VERIFICATION GATE")
print(f"{'='*65}")
print(f"")
print(f"{'Source':<25} {'Docs':>8} {'Est. Tokens':>14} {'%':>6}")
print(f"{'-'*55}")
print(f"{'Wiki_Chat':<25} {len(wiki_chat_texts):>8,} {actual_wiki_tokens:>14,} {actual_wiki_tokens/est_total*100:>5.1f}%")
print(f"{'Sadhguru articles':<25} {len(sadhguru_texts):>8,} {est_sadhguru_tokens:>14,} {est_sadhguru_tokens/est_total*100:>5.1f}%")
print(f"{'Chat replay':<25} {len(chat_replay_texts):>8,} {actual_chat_tokens:>14,} {actual_chat_tokens/est_total*100:>5.1f}%")
print(f"{'WikiHow':<25} {len(wikihow_texts):>8,} {actual_wh_tokens:>14,} {actual_wh_tokens/est_total*100:>5.1f}%")
print(f"{'Classical lit':<25} {len(classical_texts):>8,} {est_classical_tokens:>14,} {est_classical_tokens/est_total*100:>5.1f}%")
print(f"{'-'*55}")
print(f"{'TOTAL':<25} {len(all_prose_texts) + len(all_chat_texts):>8,} {est_total:>14,} {'100.0%':>6}")
print(f"")
print(f"   Avg Tamil % (prose sample): {avg_tamil:.1%}")
print(f"   Min Tamil % (prose sample): {min(sample_pcts):.1%}")
print(f"   Chat replay % of total:     {actual_chat_tokens/est_total:.1%}")
print(f"")

# Fail-fast checks
assert avg_tamil >= 0.90, f"STOP: Average Tamil {avg_tamil:.1%} < 90%"
print(f"\u2705 Average Tamil >= 90% check passed")

assert est_total >= 20_000_000, f"STOP: Only {est_total:,} estimated tokens (need >= 20M)"
print(f"\u2705 Token budget >= 20M check passed")

chat_pct = actual_chat_tokens / est_total
assert chat_pct >= 0.05, f"STOP: Chat replay only {chat_pct:.1%} (need >= 5%)"
print(f"\u2705 Chat replay >= 5% check passed")

assert chat_pct <= 0.20, f"STOP: Chat replay {chat_pct:.1%} (need <= 20%)"
print(f"\u2705 Chat replay <= 20% check passed")

print(f"\n\u2705 All quality gates passed. Proceeding to combine + pack.")

📊 QUALITY VERIFICATION GATE

Source                        Docs    Est. Tokens      %
-------------------------------------------------------
Wiki_Chat                   13,649     27,247,265  69.9%
Sadhguru articles              561      3,877,822   9.9%
Chat replay                  3,607      4,233,178  10.9%
WikiHow                        606      3,289,273   8.4%
Classical lit                1,844        359,635   0.9%
-------------------------------------------------------
TOTAL                       20,267     39,007,173 100.0%

   Avg Tamil % (prose sample): 97.6%
   Min Tamil % (prose sample): 90.8%
   Chat replay % of total:     10.9%

✅ Average Tamil >= 90% check passed
✅ Token budget >= 20M check passed
✅ Chat replay >= 5% check passed
✅ Chat replay <= 20% check passed

✅ All quality gates passed. Proceeding to combine + pack.


In [13]:
# Cell 13 — Combine & Shuffle All Sources

all_texts = wiki_chat_texts + sadhguru_texts + wikihow_texts + classical_texts + chat_replay_texts

# Shuffle with fixed seed for reproducibility
random.seed(RANDOM_SEED)
random.shuffle(all_texts)

total_chars = sum(len(t) for t in all_texts)

print(f"\U0001f4ca Combined corpus:")
print(f"   Total documents:  {len(all_texts):,}")
print(f"   Total characters: {total_chars:,}")
print(f"   Breakdown:")
print(f"     Wiki_Chat:      {len(wiki_chat_texts):,} docs")
print(f"     Sadhguru:       {len(sadhguru_texts):,} docs")
print(f"     WikiHow:        {len(wikihow_texts):,} docs")
print(f"     Classical:      {len(classical_texts):,} docs")
print(f"     Chat replay:    {len(chat_replay_texts):,} docs")

📊 Combined corpus:
   Total documents:  20,267
   Total characters: 36,393,130
   Breakdown:
     Wiki_Chat:      13,649 docs
     Sadhguru:       561 docs
     WikiHow:        606 docs
     Classical:      1,844 docs
     Chat replay:    3,607 docs


In [14]:
# Cell 14 — Pack into 1024-Token Blocks

print(f"\U0001f4e6 Packing {len(all_texts):,} docs into {BLOCK_SIZE}-token blocks...")

all_token_ids = []
eos_id = tokenizer.eos_token_id

for i, text in enumerate(all_texts):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    all_token_ids.extend(tokens)
    all_token_ids.append(eos_id)

    if (i + 1) % 5000 == 0:
        print(f"   ...tokenized {i + 1:,}/{len(all_texts):,} docs "
              f"({len(all_token_ids):,} tokens so far)")

print(f"   Total token stream: {len(all_token_ids):,} tokens")

# Split into fixed-length blocks
n_blocks = len(all_token_ids) // BLOCK_SIZE
trimmed = all_token_ids[:n_blocks * BLOCK_SIZE]
blocks = [trimmed[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE] for i in range(n_blocks)]

total_tokens = len(blocks) * BLOCK_SIZE
discarded = len(all_token_ids) - len(trimmed)

print(f"\n\u2705 Packed into {len(blocks):,} blocks of {BLOCK_SIZE} tokens")
print(f"   Total training tokens: {total_tokens:,}")
print(f"   Discarded tail:        {discarded:,} tokens")
print(f"   Efficiency:            {total_tokens / len(all_token_ids):.1%}")

📦 Packing 20,267 docs into 1024-token blocks...
   ...tokenized 5,000/20,267 docs (9,865,034 tokens so far)
   ...tokenized 10,000/20,267 docs (19,862,929 tokens so far)
   ...tokenized 15,000/20,267 docs (29,376,713 tokens so far)
   ...tokenized 20,000/20,267 docs (39,008,174 tokens so far)
   Total token stream: 39,506,761 tokens

✅ Packed into 38,580 blocks of 1024 tokens
   Total training tokens: 39,505,920
   Discarded tail:        841 tokens
   Efficiency:            100.0%


In [15]:
# Cell 15 — Block Quality Verification

print(f"\U0001f50d Verifying block quality (10 random samples)...\n")

random.seed(RANDOM_SEED)
check_indices = random.sample(range(len(blocks)), min(10, len(blocks)))

for idx in check_indices:
    decoded = tokenizer.decode(blocks[idx])
    pct = tamil_char_pct(decoded)
    is_chat = '<|im_start|>' in decoded
    tag = " [chat]" if is_chat else ""
    print(f"   Block {idx:>6}: Tamil {pct:.0%}{tag} | {decoded[:120]}...")

# Full distribution (sample 500 blocks)
n_sample = min(500, len(blocks))
print(f"\n\U0001f4ca Tamil % distribution across {n_sample} sampled blocks:")
sample_block_indices = random.sample(range(len(blocks)), n_sample)
all_block_pcts = []
for idx in sample_block_indices:
    decoded = tokenizer.decode(blocks[idx])
    all_block_pcts.append(tamil_char_pct(decoded))

pct_array = np.array(all_block_pcts)
buckets = [(0, 0.5), (0.5, 0.7), (0.7, 0.8), (0.8, 0.9), (0.9, 1.01)]
for lo, hi in buckets:
    count = np.sum((pct_array >= lo) & (pct_array < hi))
    bar = '#' * int(count / len(pct_array) * 50)
    print(f"   {lo:.0%}-{hi:.0%}: {count:>4} ({count/len(pct_array):>5.1%}) {bar}")

print(f"\n   Mean:   {np.mean(all_block_pcts):.1%}")
print(f"   Median: {np.median(all_block_pcts):.1%}")
print(f"   Min:    {np.min(all_block_pcts):.1%}")

low_blocks = np.sum(pct_array < 0.50)
if low_blocks > len(pct_array) * 0.15:
    print(f"   \u26a0\ufe0f {low_blocks} blocks < 50% Tamil — check data mix")
else:
    print(f"   \u2705 Block quality looks good ({low_blocks} blocks < 50% Tamil)")

🔍 Verifying block quality (10 random samples)...

   Block   7296: Tamil 97% | �்கள் குளிர்சாதன பெட்டியின் மிருதுவான இழுப்பறையில் 1 முதல் 2 வாரங்களுக்கு ரப்பர்ப்பை சேமிக்கவும்.
4. நீங்கள் பயன்படுத்தத...
   Block   1639: Tamil 95% | ும் வேலை செய்யாது.

2. "புதிய செய்தி" பொத்தானைத் தட்டவும்.
இது திரையின் கீழ்-வலது மூலையில் உள்ள பென்சில் ஐகான் ஆகும்.

3...
   Block  18024: Tamil 97% | னவிலங்குகளைப் பாதுகாக்க, பல பாதுகாப்பு முயற்சிகள் மேற்கொள்ளப்பட்டுள்ளனஃ

1. வாழ்விடப் பாதுகாப்புஃ சுற்றுச்சூழல் அமைப்பின...
   Block  16049: Tamil 97% | ர்ப்பது, அவர்களுடைய கல்லூரி படிப்பு, அவர்களுக்கு திருமணம், அது இது என்று அவர்கள் அங்கே போகவே இல்லை. 75 வயதிற்கு மேல்தான்...
   Block  14628: Tamil 96% | ுயினோவா
வேகவைத்த ப்ரோக்கோலி
பச்சை பீன்ஸ்
சிற்றுண்டிஃ

பழம் மற்றும் பாதாம் பாலுடன் புரதம் கலக்கவும்
நினைவில் கொள்ளுங்கள்,...
   Block   9144: Tamil 97% | �்சாலையில் ஒரு நாயாகட்டும், அதன் மரணம் என்னை மிகவும் பாதிக்கிறது. நான் ஏன் இப்படி உணருகிறேன்?

சத்குரு: அனைத்து பயங்களுக...
   Block   6717: Tamil 97% | ார்

In [16]:
# Cell 16 — Upload to HuggingFace

from datasets import Dataset

packed_dataset = Dataset.from_dict({
    "input_ids": blocks,
    "attention_mask": [[1] * BLOCK_SIZE for _ in blocks],
    "labels": [list(b) for b in blocks],
})

print(f"\U0001f4ca Dataset created:")
print(f"   Blocks:       {len(packed_dataset):,}")
print(f"   Total tokens: {len(packed_dataset) * BLOCK_SIZE:,}")
print(f"   Columns:      {packed_dataset.column_names}")

print(f"\n\U0001f4e4 Uploading to {HF_DATASET_OUT}...")

sources_desc = (
    f"WikiChat:{len(wiki_chat_texts)}, "
    f"Sadhguru:{len(sadhguru_texts)}, "
    f"ChatReplay:{len(chat_replay_texts)}, "
    f"WikiHow:{len(wikihow_texts)}, "
    f"Classical:{len(classical_texts)}"
)

packed_dataset.push_to_hub(
    HF_DATASET_OUT,
    private=False,
    commit_message=(
        f"DAPT v2.1: {len(blocks):,} blocks x {BLOCK_SIZE} tokens = {total_tokens:,} | "
        f"Sources: {sources_desc} | "
        f"Tamil >= {TAMIL_THRESHOLD:.0%} | NFKC cleaned"
    ),
)

print(f"\n\u2705 Dataset uploaded: https://huggingface.co/datasets/{HF_DATASET_OUT}")

# Verify upload
print(f"\n\U0001f50d Verifying upload...")
verify_ds = load_dataset(HF_DATASET_OUT)
print(f"   Blocks: {len(verify_ds['train']):,}")
sample = verify_ds['train'][0]
assert len(sample['input_ids']) == BLOCK_SIZE, "Block size mismatch!"
print(f"   \u2705 Verification passed")

📊 Dataset created:
   Blocks:       38,580
   Total tokens: 39,505,920
   Columns:      ['input_ids', 'attention_mask', 'labels']

📤 Uploading to CryptoYogi/vazhi-dapt-tamil-v2_1...


Uploading the dataset shards:   0%|          | 0/2 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

...

README.md:   0%|          | 0.00/356 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/33.1M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/32.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/38580 [00:00<?, ? examples/s]

   Blocks: 38,580
   ✅ Verification passed


In [17]:
# Cell 17 — Summary

print(f"{'='*65}")
print(f"\U0001f4cb DAPT DATA PREP v2.1 — SUMMARY")
print(f"{'='*65}")
print(f"")
print(f"   Dataset:         {HF_DATASET_OUT}")
print(f"   Total blocks:    {len(blocks):,}")
print(f"   Block size:      {BLOCK_SIZE} tokens")
print(f"   Total tokens:    {total_tokens:,}")
print(f"   Tamil threshold: >= {TAMIL_THRESHOLD:.0%}")
print(f"")
print(f"   Sources:")
print(f"     Wiki_Chat:        {len(wiki_chat_texts):>6,} docs (~{actual_wiki_tokens:,} tokens, {actual_wiki_tokens/est_total*100:.0f}%)")
print(f"     Sadhguru:         {len(sadhguru_texts):>6,} docs (~{est_sadhguru_tokens:,} tokens, {est_sadhguru_tokens/est_total*100:.0f}%)")
print(f"     Chat replay:      {len(chat_replay_texts):>6,} docs (~{actual_chat_tokens:,} tokens, {actual_chat_tokens/est_total*100:.0f}%)")
print(f"     WikiHow:          {len(wikihow_texts):>6,} docs (~{actual_wh_tokens:,} tokens, {actual_wh_tokens/est_total*100:.0f}%)")
print(f"     Classical lit:    {len(classical_texts):>6,} docs (~{est_classical_tokens:,} tokens, {est_classical_tokens/est_total*100:.0f}%)")
print(f"")
print(f"   Key improvements over v2.0:")
print(f"     Token budget:     {total_tokens:,} vs 4,795,392 ({total_tokens/4795392:.1f}x)")
print(f"     Chat replay:      {actual_chat_tokens/est_total:.1%} vs 1.4%")
print(f"     Source diversity:  5 sources vs 3")
print(f"")
print(f"\U0001f449 Next: Create + run DAPT v2.1 training notebook on Colab Pro (GPU)")
print(f"   Base model: Qwen/Qwen3-0.6B (vanilla instruct)")
print(f"   Dataset:    {HF_DATASET_OUT}")
print(f"   Strategy:   Multi-epoch with interim eval gates (from v2.0 lesson)")

📋 DAPT DATA PREP v2.1 — SUMMARY

   Dataset:         CryptoYogi/vazhi-dapt-tamil-v2_1
   Total blocks:    38,580
   Block size:      1024 tokens
   Total tokens:    39,505,920
   Tamil threshold: >= 90%

   Sources:
     Wiki_Chat:        13,649 docs (~27,247,265 tokens, 70%)
     Sadhguru:            561 docs (~3,877,822 tokens, 10%)
     Chat replay:       3,607 docs (~4,233,178 tokens, 11%)
     WikiHow:             606 docs (~3,289,273 tokens, 8%)
     Classical lit:     1,844 docs (~359,635 tokens, 1%)

   Key improvements over v2.0:
     Token budget:     39,505,920 vs 4,795,392 (8.2x)
     Chat replay:      10.9% vs 1.4%
     Source diversity:  5 sources vs 3

👉 Next: Create + run DAPT v2.1 training notebook on Colab Pro (GPU)
   Base model: Qwen/Qwen3-0.6B (vanilla instruct)
   Dataset:    CryptoYogi/vazhi-dapt-tamil-v2_1
   Strategy:   Multi-epoch with interim eval gates (from v2.0 lesson)
